# 05 · Evaluation and Final Results — SOL-USD
**Data Science Diploma · ENES UNAM León**

This notebook generates the visualizations and final analysis for the **technical report** (section 7. Results).

### Metrics Used
| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | mean(\|y - ŷ\|) | Average error in USD — easy to communicate |
| **RMSE** | √mean((y-ŷ)²) | Penalizes large errors — more demanding |
| **MAPE** | mean(\|y-ŷ\|/y)×100 | Error in % — scale-independent |
| **R²** | 1 - SS_res/SS_tot | How much variance the model explains (0-1) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Base paths
RAW_DIR       = os.path.join(os.getcwd(), 'data', 'raw')
PROCESSED_DIR = os.path.join(os.getcwd(), 'data', 'processed')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load saved predictions
df_preds = pd.read_csv(os.path.join(PROCESSED_DIR, 'predicciones_test.csv'), index_col=0, parse_dates=True)
print(df_preds.head())

## 5.1 Final Metrics (Summary Table)

In [ ]:
def metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    return {'MAE ($)': round(mae,2), 'RMSE ($)': round(rmse,2), 'MAPE (%)': round(mape,2), 'R²': round(r2,4)}

table = pd.DataFrame({
    'Ridge Regression': metrics(df_preds['real'], df_preds['pred_ridge']),
    'XGBoost':          metrics(df_preds['real'], df_preds['pred_xgb'])
}).T

print('\n===== FINAL METRICS — TEST SET =====')
print(table.to_string())
table

## 5.2 Main Chart: Prediction vs Actual

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, model, color, title in zip(
    axes,
    ['pred_ridge', 'pred_xgb'],
    ['#FFB800', '#14F195'],
    ['Ridge Regression', 'XGBoost']
):
    ax.plot(df_preds.index, df_preds['real'], label='Actual', color='#9945FF', linewidth=2)
    ax.plot(df_preds.index, df_preds[model], label=f'{title} Prediction', color=color,
            linewidth=1.5, linestyle='--', alpha=0.9)

    # ±MAE error band
    mae_val = mean_absolute_error(df_preds['real'], df_preds[model])
    ax.fill_between(df_preds.index,
                    df_preds[model] - mae_val,
                    df_preds[model] + mae_val,
                    alpha=0.15, color=color, label=f'±MAE Band (${mae_val:.1f})')
    ax.set_title(f'{title} — Prediction vs Actual', fontsize=12)
    ax.set_ylabel('USD')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(RAW_DIR, '05_final_prediction.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Residual Analysis

A good model has residuals centered at 0, without patterns, and normally distributed.

In [ ]:
df_preds['residual_ridge'] = df_preds['real'] - df_preds['pred_ridge']
df_preds['residual_xgb']   = df_preds['real'] - df_preds['pred_xgb']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (model, color, name) in enumerate([
    ('residual_ridge', '#FFB800', 'Ridge'),
    ('residual_xgb',   '#14F195', 'XGBoost')
]):
    # Residuals over time
    axes[i, 0].plot(df_preds.index, df_preds[model], color=color, linewidth=1, alpha=0.8)
    axes[i, 0].axhline(0, color='red', linewidth=1, linestyle='--')
    axes[i, 0].set_title(f'{name} — Residuals Over Time')
    axes[i, 0].set_ylabel('Error (USD)')

    # Residual histogram
    axes[i, 1].hist(df_preds[model], bins=25, color=color, edgecolor='white', alpha=0.85)
    axes[i, 1].axvline(0, color='red', linewidth=1.5, linestyle='--')
    axes[i, 1].axvline(df_preds[model].mean(), color='white', linewidth=1.5,
                       label=f'Mean: ${df_preds[model].mean():.2f}')
    axes[i, 1].set_title(f'{name} — Residual Distribution')
    axes[i, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RAW_DIR, '05_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Scatter: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, color, name in zip(
    axes,
    ['pred_ridge', 'pred_xgb'],
    ['#FFB800', '#14F195'],
    ['Ridge Regression', 'XGBoost']
):
    ax.scatter(df_preds['real'], df_preds[model], color=color, alpha=0.6, s=30)

    # Perfect prediction line
    lims = [min(df_preds['real'].min(), df_preds[model].min()),
            max(df_preds['real'].max(), df_preds[model].max())]
    ax.plot(lims, lims, color='red', linewidth=1.5, linestyle='--', label='Perfect Prediction')

    r2 = r2_score(df_preds['real'], df_preds[model])
    ax.set_title(f'{name} | R² = {r2:.4f}')
    ax.set_xlabel('Actual (USD)')
    ax.set_ylabel('Predicted (USD)')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RAW_DIR, '05_scatter_actual_pred.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.5 Trading Signal Simulation

An intuitive way to present results in the video: does the model correctly predict the direction of movement?

In [ ]:
# Actual vs predicted direction
prev_close = df_preds['real'].shift(1)

real_dir = np.sign(df_preds['real'] - prev_close)          # +1 up, -1 down
pred_ridge_dir = np.sign(df_preds['pred_ridge'] - prev_close)
pred_xgb_dir   = np.sign(df_preds['pred_xgb']   - prev_close)

acc_ridge = (real_dir == pred_ridge_dir).mean() * 100
acc_xgb   = (real_dir == pred_xgb_dir).mean()   * 100

print('===== Directional Accuracy (up or down?) =====')
print(f'Ridge Regression : {acc_ridge:.1f}%')
print(f'XGBoost          : {acc_xgb:.1f}%')
print(f'\n(Random baseline: ~50%)')
print(f'If the model exceeds 55% directional accuracy in crypto, it is significant.')

## 5.6 Final Summary Table for the Report

In [ ]:
final_table = pd.DataFrame({
    'Model': ['Ridge Regression (baseline)', 'XGBoost'],
    'MAE ($)':  [
        round(mean_absolute_error(df_preds['real'], df_preds['pred_ridge']), 2),
        round(mean_absolute_error(df_preds['real'], df_preds['pred_xgb']), 2)
    ],
    'RMSE ($)': [
        round(np.sqrt(mean_squared_error(df_preds['real'], df_preds['pred_ridge'])), 2),
        round(np.sqrt(mean_squared_error(df_preds['real'], df_preds['pred_xgb'])), 2)
    ],
    'MAPE (%)': [
        round(np.mean(np.abs((df_preds['real'] - df_preds['pred_ridge']) / df_preds['real'])) * 100, 2),
        round(np.mean(np.abs((df_preds['real'] - df_preds['pred_xgb'])   / df_preds['real'])) * 100, 2)
    ],
    'R²': [
        round(r2_score(df_preds['real'], df_preds['pred_ridge']), 4),
        round(r2_score(df_preds['real'], df_preds['pred_xgb']), 4)
    ],
    'Dir. Accuracy (%)': [round(acc_ridge, 1), round(acc_xgb, 1)]
})

print('\n' + '='*60)
print('FINAL TABLE — COPY THIS TO THE TECHNICAL REPORT (Section 7)')
print('='*60)
print(final_table.to_string(index=False))

final_table.to_csv(os.path.join(PROCESSED_DIR, 'final_results_table.csv'), index=False)
print('\nTable saved ✓')

---
## ✅ Project Complete

### Generated Files
```
data/
  raw/
    sol_usd_raw.csv                  # Raw historical data
    01_price_volume.png
    02_distribution.png
    02_returns_volatility.png
    02_correlation.png
    02_acf_pacf.png
    02_monthly.png
    03_*.png                         # Feature visualizations
    04_*.png                         # Model visualizations
    05_*.png                         # Final evaluation
  processed/
    sol_usd_eda.csv
    sol_usd_features.csv
    predicciones_test.csv
    ridge_model.pkl
    scaler.pkl
    xgboost_model.json
    final_results_table.csv          ← For the technical report
```

### For the Video (10-15 min)
1. **Problem** (2 min): Why predict Solana? Crypto context
2. **Data** (2 min): Show the price and volume chart
3. **Methodology** (3 min): Features created, temporal split, models
4. **Results** (4 min): Prediction vs actual, MAPE, directional accuracy
5. **Conclusions** (2 min): What works? What's next? (LSTM, more features)

### Next Steps (if time allows)
- Add LSTM for comparison with deep learning
- Walk-forward validation (more robust for time series)
- Hyperparameter tuning with Optuna